# The final CAT indicator dataset — replication report

One page of plots and tables for `derived/north_atlantic_audit` +
`thresholds_2026-09-11.json`, the dataset selected in `FINAL_DATASET.md`.

**This notebook computes nothing from ERA5.** It reads the archived CSV
reductions and renders. The figure code lives in `ada/prosser_figures.py` and
`ada/final_figures.py`, imported rather than copied, so a figure here can never
disagree with the same figure produced from the command line:

```bash
python ada/prosser_figures.py --outputs cat_outputs/final   # the 21-panel layouts
python ada/final_figures.py  --outputs cat_outputs/final    # the three summaries
```

**Inputs it looks for, in order:**

| | directory | tag | what it is |
|---|---|---|---|
| 1 | `cat_outputs/final/` | `_audit` | the final dataset, LOG…SOG (job 1124312) |
| 2 | `cat_outputs/` | `_audit` | an audit table pulled on its own |
| 3 | `data_prosser/` | *(none)* | **the baseline series** — the appendix sensitivity, MOG and SOG only |

If it falls through to 3, every number below is the *baseline* convention set,
which is not the dataset. The banner says which was used — read it.

In [ ]:
from pathlib import Path
import sys, csv

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ada').is_dir())
sys.path.insert(0, str(REPO / 'ada'))
FIGDIR = REPO / 'cat_outputs' / 'figures_report'
FIGDIR.mkdir(parents=True, exist_ok=True)

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import prosser_figures as pf
import final_figures as ff
from prosser_published import FIGURE4

CANDIDATES = [(REPO / 'cat_outputs' / 'final', '_audit'),
              (REPO / 'cat_outputs', '_audit'),
              (REPO / 'data_prosser', '')]
OUTPUTS, TAG = next(((d, t) for d, t in CANDIDATES
                     if any(d.glob(f'per_diagnostic_annual_*{t}_series.csv'))),
                    (None, None))
assert OUTPUTS, 'no per_diagnostic_annual_*_series.csv anywhere — pull the CSVs first'

SERIES = ('FINAL (audit conventions: F1 + F12 + F6)' if 'audit' in TAG
          else 'BASELINE (appendix sensitivity) — NOT the final dataset')
found = sorted({p.name for p in OUTPUTS.glob(f'per_diagnostic_annual_*{TAG}_series.csv')})
print(f'series   : {SERIES}')
print(f'directory: {OUTPUTS}')
print(f'tables   : ' + ', '.join(f.split('annual_')[1].split(TAG or "_series")[0] for f in found))
print(f'figures  : {FIGDIR}')

## 1. Does it replicate Prosser at the ensemble level?

Prosser's Table 1 is the only thing he tabulates, so this comparison is exact.
The ensemble is the mean of the 21 exceedance fields, as in his method.

In [ ]:
rows = []
for sev in ['light', 'light_to_moderate', 'moderate', 'moderate_to_severe', 'severe']:
    p = OUTPUTS / f'per_diagnostic_annual_{sev}{TAG}_series.csv'
    if not p.exists():
        continue
    years, data = pf.read_series(p)
    s = pf.fit_summary(years, data['ensemble'])
    his = pf.PROSSER_ANNUAL_REL[sev]
    rows.append(dict(severity=pf.SEVERITY_LABEL[sev],
                     ours=f"{s['rel']:+.0%}", ci=f"[{s['lo']:+.0%}, {s['hi']:+.0%}]",
                     prosser=f'{his:+.0%}', ratio=round(s['rel'] / his, 2),
                     t=round(s['fit']['t'], 2),
                     contains_his='yes' if s['lo'] <= his <= s['hi'] else 'NO',
                     level_1979=f"{s['y0']:.3%}",
                     his_level_1979=f"{pf.PROSSER_ANNUAL_HOURS_1979[sev] / 8760:.3%}"))
try:
    import pandas as pd
    display(pd.DataFrame(rows).set_index('severity'))
except ImportError:
    for r in rows:
        print(r)

## 2. The published figures, panel for panel

`ada/prosser_figures.py` writes Figure 3a (the ensemble) and the 21-panel
Figure 4 / S4-LOG / S4-SOG layouts in **his** order, with **his** titles, and
— at MOG, where his axis limits were transcribed — on **his** axes, so the two
can be laid side by side. Under every panel: his printed `rel`/`abs`/`p`, then
ours computed the same way.

A panel drawn with a red box does not fit on his axis. That is a finding, not a
plotting failure: it means the **level** disagrees.

Running the renderer also prints the cross-check against the archived fit CSVs —
silence there means the picture and the table agree about the same run.

In [ ]:
import subprocess
cmd = [sys.executable, str(REPO / 'ada' / 'prosser_figures.py'),
       '--outputs', str(OUTPUTS), '--figdir', str(FIGDIR), '--season', 'annual']
print(subprocess.run(cmd, capture_output=True, text=True).stdout)

In [ ]:
from IPython.display import Image, display
for sev in ['mog', 'log', 'sog']:
    for axes in ['his_axes', 'our_axes']:
        p = FIGDIR / f'fig4_prosser_layout_{sev}{TAG}_{axes}.png'
        if p.exists():
            print(p.name)
            display(Image(filename=str(p), width=950))
            break

## 3. The three summaries

The panels show 21 pictures; these show the 63 comparisons at once.

1. **`final_fit_vs_published`** — our fitted change against his printed change,
   one point per diagnostic per severity, our 95 % interval as a bar. On the
   1:1 line means we reproduce his number; a bar crossing the line contains it.
   Only the cells that do not are labelled.
2. **`final_level_ratio`** — the level gap per diagnostic, log scale, with the
   diagnostics that use the 175/225 hPa vertical stencil marked `·S`. This is
   the stencil argument in one picture.
3. **`final_tail_latitude`** — where each diagnostic's global tail lives in the
   reference year (needs `tail_latitude_2000_audit.csv` from step 1b of the
   batch). `ubf` is the one whose extremes are polar rather than jet-aligned.

In [ ]:
data = ff.load(OUTPUTS, TAG)
print(ff.figure_fit_vs_published(data, FIGDIR, TAG).name)
print(ff.figure_level_ratio(data, FIGDIR, TAG).name)
tail = OUTPUTS / f'tail_latitude_2000{TAG}.csv'
for name in [f'final_fit_vs_published{TAG}.png', f'final_level_ratio{TAG}.png']:
    display(Image(filename=str(FIGDIR / name), width=820))
if tail.exists():
    for sev in ['light', 'severe']:
        p = ff.figure_tail_latitude(tail, FIGDIR, TAG, sev)
        display(Image(filename=str(p), width=820))
else:
    print(f'{tail.name} not here — run step 1b of jobs/27 and pull it to see the zonal profiles')

## 4. The table view

Every number in the figures above, so nothing depends on reading a colour.

In [ ]:
rows = ff.table_rows(data)
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df.pivot_table(index='diagnostic', columns='severity',
                           values=['ratio', 'change', 'inside'], aggfunc='first'))
    display(df)
except ImportError:
    for r in rows:
        print(r)

## 5. How to read the ones that do not fit

Four questions, asked in this order, settle every disagreement in the table
above. The classes are `FINAL_DATASET.md` §5; the evidence is
`BATCH_RESULTS_2026-09-21.md`.

**1. Is the TREND inside our interval?** If yes, the diagnostic replicates the
quantity the paper is about, whatever its level does. 59 of 63 cells pass, and
21 of 21 at LOG. A level gap with an agreeing trend is a calibration offset,
not a broken diagnostic.

**2. If the level disagrees, does it use the vertical stencil?** Ours is
175/225 hPa (50 hPa); his is model levels 73–75 (≈18 hPa). The stencil group
carries most of the ensemble level gap and its median ratio sits well below the
single-level group's. `endlich`, `ngm2`, `colson_panofsky`, `negative_richardson`
and `f2d` are all in that group, and none of them can be closed without MARS
model levels. Report the number; do not chase it.

**3. If it disagrees and is single-level, is the disagreement in the tail
only?** `ncsu1` agrees with Prosser at LOG (+20 % against +21 %) and departs
above it, because A36's `1/max(Ri, 1e-5)` floor binds often on a stencil that
resolves thin unstable layers and almost never on ours. Two different
populations above LOG, one shared population at LOG.

**4. Is it a numerical property of our own implementation?** That is `ubf`, and
it is the only one. Its extreme tail is polar where every other diagnostic's is
jet-aligned, which points at the conditioning of a near-cancelling residual at
high latitude, not at the atmosphere. Ensemble member only; do not use its
magnitudes.

**What does not count as an answer:** a significance call that flips. With
n = 42 and a sparse diagnostic, `colson_panofsky` reaches +62 % with an interval
that contains his +45 % and still misses significance, because our North
Atlantic box holds seven times fewer of its events than his does. That is
power, not disagreement.

In [ ]:
CLASSES = {
    'A — no caveat': ['vertical_wind_shear', 'brown1', 'brown2', 'ti1', 'ti2',
                      'deformation', 'vorticity_squared', 'temperature_gradient',
                      'wind_speed', 'ngm1', 'nva', 'rva_magnitude',
                      'horizontal_divergence'],
    'B — stated caveat': ['magnitude_pv', 'ngm2'],
    'C — ensemble member only': ['negative_richardson', 'colson_panofsky', 'f2d',
                                 'ubf', 'ncsu1', 'endlich'],
}
assert sum(map(len, CLASSES.values())) == 21

sev = 'moderate' if 'moderate' in data else next(iter(data))
for cls, names in CLASSES.items():
    print(f'\n{cls}')
    for n in sorted(names, key=lambda k: data[sev][k]['ratio']):
        r = data[sev][n]
        print(f"   {n:<22} level x{r['ratio']:.2f}   ours {r['change']:+.0%} "
              f"[{r['lo']:+.0%}, {r['hi']:+.0%}]   his {r['his_rel']:+.0%}   "
              f"{'contains his' if r['inside'] else 'OUTSIDE'}")

# what the ensemble owes to the class-C members
years, series = pf.read_series(OUTPUTS / f'per_diagnostic_annual_{sev}{TAG}_series.csv')
names21 = [c for c in series if c != 'ensemble']
keep = [n for n in names21 if n not in CLASSES['C — ensemble member only']]
all21 = pf.fit_summary(years, np.mean([series[n] for n in names21], axis=0))['rel']
woC = pf.fit_summary(years, np.mean([series[n] for n in keep], axis=0))['rel']
print(f'\nensemble change, mean of 21: {all21:+.1%}; without the six class-C '
      f'members: {woC:+.1%}  (difference {100 * (woC - all21):+.1f} pp)')

## 6. What this does not show

- **Anything outside 36–60°N, 55–10°W.** The global maps (his Figures 1, 2, S5)
  and the USA box (3b) were never computed; they need new downloads. The
  zonal-profile figure is the one spatial check that the reference year already
  on disk can support.
- **Seasonal per-diagnostic cuts.** He publishes no per-diagnostic seasonal
  numbers, so those fits have no target to be scored against.
- **Magnitudes against published tables.** `brown2` is missing an unstated
  length², `magnitude_pv` carries ×100 from the hPa coordinate and is A18 rather
  than Ertel PV, `colson_panofsky` is in units of λ². All three are constants
  and cancel out of every exceedance decision, and none of them can be compared
  to a published magnitude.

`FINAL_DATASET.md` §3 has each limitation with its measured impact.